**LENDING CLUB LOAN DATA ANALİZİ **


**LendingClub Loan Data Analizi**

**1. Adım : Dataseti Colaba Yükleme ve Gereksiz Olanları Temizleme**

In [ ]:
import pandas as pd
import numpy as np

# Dosya yolunu klasör adını ekleyerek güncelledik:
path = '/content/drive/MyDrive/Bitirme_Projesi/accepted_2007_to_2018Q4.csv'

try:
    print("Veri yükleniyor, lütfen bekleyin...")
    # İlk 100 bin satırı yüklüyoruz
    df = pd.read_csv(path, low_memory=False, nrows=100000)
    print("✅ BAŞARILI: Veri yüklendi!")

    # 1. Adım: Boş sütunları temizle (%50'den fazlası boş olanlar)
    missing_fractions = df.isnull().mean()
    drop_cols = missing_fractions[missing_fractions > 0.5].index
    df.drop(columns=drop_cols, inplace=True)

    # 2. Adım: Gereksiz sütunları çıkar
    # DİKKAT: 'id' sütununu buradan çıkardım, böylece BigQuery tabloların bozulmayacak!
    extra_drop = ['url', 'title', 'zip_code', 'policy_code', 'desc', 'member_id']
    existing_extra_drop = [c for c in extra_drop if c in df.columns]
    df.drop(columns=existing_extra_drop, inplace=True)

    print(f"✨ Temizlik bitti! Kalan sütun sayısı: {len(df.columns)}")
    print(f"🆔 'id' sütunu korundu, BigQuery entegrasyonu hazır.")
    display(df.head())

except Exception as e:
    print(f"❌ Bir hata oluştu: {e}")

Veri yükleniyor, lütfen bekleyin...
✅ BAŞARILI: Veri yüklendi!
✨ Temizlik bitti! Kalan sütun sayısı: 90
🆔 'id' sütunu korundu, BigQuery entegrasyonu hazır.


,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,debt_settlement_flag
0,68407277,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,...,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,N,Cash,N
1,68355089,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,...,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,N,Cash,N
2,68341763,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,truck driver,...,50.0,0.0,0.0,218418.0,18696.0,6200.0,14877.0,N,Cash,N
3,66310712,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,Information Systems Officer,...,0.0,0.0,0.0,381215.0,52226.0,62500.0,18000.0,N,Cash,N
4,68476807,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,Contract Specialist,...,60.0,0.0,0.0,439570.0,95768.0,20300.0,88097.0,N,Cash,N


**2.Adım: Veri Tiplerini Düzeltme (Format Sorunları)**

In [ ]:
import pandas as pd
import numpy as np

# 1. 'int_rate' temizliği (Eğer daha önce yapmadıysan)
if 'int_rate' in df.columns:
    df['int_rate'] = df['int_rate'].astype(str).str.replace('%', '').astype(float)

# 2. 'term' temizliği (Başına r ekleyerek Regex hatasını çözdük)
if 'term' in df.columns:
    df['term'] = df['term'].astype(str).str.extract(r'(\d+)').astype(float)

# 3. 'emp_length' temizliği (TypeError hatasını str(val) ile çözdük)
def clean_emp_length(val):
    val = str(val).lower() # Değeri metne çevir ve küçük harf yap
    if val == 'nan' or val == 'none' or val == '':
        return 0
    if '10+' in val:
        return 10
    if '< 1' in val:
        return 0
    # İçindeki rakamları ayıkla
    digits = ''.join(filter(str.isdigit, val))
    return int(digits) if digits else 0

df['emp_length'] = df['emp_length'].apply(clean_emp_length)

print("✅ Format dönüşümleri (Regex ve Type hataları giderilerek) tamamlandı.")
print(df[['int_rate', 'term', 'emp_length']].head())

✅ Format dönüşümleri (Regex ve Type hataları giderilerek) tamamlandı.
   int_rate  term  emp_length
0     13.99  36.0          10
1     11.99  36.0          10
2     10.78  60.0          10
3     14.85  60.0          10
4     22.45  60.0           3


**3. Adım: Veriyi 5 Mantıksal Tabloya Bölme**

In [ ]:
# --- ÖNCE EKSİK SÜTUNLARI OLUŞTURMA ---

# 1. Tarih dönüşümü (Hatalı formatları engellemek için errors='coerce' ekledik)
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')

# 2. issue_year oluşturma (NaT değerleri temizlemek için)
df['issue_year'] = df['issue_d'].dt.year

# 3. Kredi Geçmişi tarih dönüşümü
if 'earliest_cr_line' in df.columns:
    df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')

print("✅ Tarih sütunları ve issue_year başarıyla oluşturuldu.")

# --- ŞİMDİ TABLOLARA AYIRMA (Looker Studio Entegrasyonu İçin) ---

# 1. Tablo: Müşteri Tanıtım Kartı
# Not: Eğer temizlik aşamasında emp_title'ı sildiysen hata almamak için kontrol ekledik
borrower_cols = ['id', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'addr_state']
if 'emp_title' in df.columns: borrower_cols.append('emp_title')
borrowers = df[borrower_cols].copy()

# 2. Tablo: Kredi Detayları
loan_details = df[['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'issue_d', 'issue_year']].copy()

# 3. Tablo: Kredi Geçmişi ve Risk
credit_risk = df[['id', 'fico_range_low', 'fico_range_high', 'earliest_cr_line', 'pub_rec', 'pub_rec_bankruptcies', 'delinq_2yrs']].copy()

# 4. Tablo: Mali Sağlık
financial_health = df[['id', 'dti', 'open_acc', 'total_acc', 'revol_bal', 'revol_util']].copy()

# 5. Tablo: Performans ve Sonuç
# Burada loan_status filtresi önemli; sadece bitmiş kredileri (Fully Paid/Charged Off) alıyoruz
loan_performance = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])][['id', 'loan_status', 'total_pymnt', 'recoveries']].copy()

print("🚀 5 tablo (id sütunu dahil) hatasız bir şekilde başarıyla oluşturuldu!")

✅ Tarih sütunları ve issue_year başarıyla oluşturuldu.
🚀 5 tablo (id sütunu dahil) hatasız bir şekilde başarıyla oluşturuldu!


**Aykırı Değer (Outlier) Temizliği**



In [ ]:
# Yıllık gelirdeki uç değerleri (örneğin %99'luk dilimin üzerini) temizledim
q_limit = borrowers['annual_inc'].quantile(0.99)
borrowers = borrowers[borrowers['annual_inc'] <= q_limit]

# Kredi miktarında da benzer bir kontrol yaptım
q_loan = loan_details['loan_amnt'].quantile(0.99)
loan_details = loan_details[loan_details['loan_amnt'] <= q_loan]

print("✅ Aykırı değerler temizlendi.")

✅ Aykırı değerler temizlendi.


**Tablolar Arası Tutarlılık Kontrolü**

In [ ]:
# Tüm tabloların ortak ID setine göre filtrelenmesi (Inner Join Mantığı)
common_ids = set(borrowers['id']) & set(loan_details['id']) & set(credit_risk['id']) & \
             set(financial_health['id']) & set(loan_performance['id'])

borrowers = borrowers[borrowers['id'].isin(common_ids)]
loan_details = loan_details[loan_details['id'].isin(common_ids)]
credit_risk = credit_risk[credit_risk['id'].isin(common_ids)]
financial_health = financial_health[financial_health['id'].isin(common_ids)]
loan_performance = loan_performance[loan_performance['id'].isin(common_ids)]

print(f"✅ Tablolar eşitlendi. Toplam ortak kayıt sayısı: {len(common_ids)}")

✅ Tablolar eşitlendi. Toplam ortak kayıt sayısı: 87005


**3. Yeni Metrikler (Calculated Fields) Üretme**
Looker'da formül yazmak bazen yavaşlatabileceği için bazı kritik finansal metrikleri burada hesaplayıp hazır gönderelim:

Kredi/Gelir Oranı: Kişinin borç yükünü anlamak için.

Net Getiri: Bankanın bu krediden kazandığı net faiz.

In [ ]:
# 1. Kredi/Gelir Oranı (Loan-to-Income Ratio)
# loan_details ve borrowers'ı geçici birleştirip hesaplayalım
temp = pd.merge(loan_details[['id', 'loan_amnt']], borrowers[['id', 'annual_inc']], on='id')
temp['loan_to_income_ratio'] = temp['loan_amnt'] / temp['annual_inc']
# Bu yeni kolonu loan_details'e geri verelim
loan_details = pd.merge(loan_details, temp[['id', 'loan_to_income_ratio']], on='id')

# 2. Geri Ödeme Oranı (%)
loan_performance['recovery_rate'] = (loan_performance['total_pymnt'] / loan_details['loan_amnt']) * 100

print("✅ Looker için yeni metrikler hesaplandı.")

✅ Looker için yeni metrikler hesaplandı.


**4. Kategorik Verilerin Standardizasyonu**

In [ ]:
# Eyalet kodlarını büyük harfe sabitleyelim
borrowers['addr_state'] = borrowers['addr_state'].str.upper()

# Ev sahipliği verisindeki 'ANY', 'NONE' gibi belirsiz değerleri 'OTHER' yapalım
borrowers['home_ownership'] = borrowers['home_ownership'].replace(['ANY', 'NONE'], 'OTHER')

print("✅ Kategorik veriler standart hale getirildi.")

✅ Kategorik veriler standart hale getirildi.


**5. Binned (Kutulanmış) Veriler Üretmek**
Looker'da FICO skoru veya Kredi Tutarı gibi sürekli sayıları gruplandırmak (Örn: 500-600 arası, 600-700 arası) zor olacağı için bunu Python'da yapıp Looker'a hazır "Etiketler" gönderme

In [ ]:
# FICO Skorlarını Gruplandırma yapıldı
bins = [0, 630, 690, 720, 850]
labels = ['Zayıf (Riskli)', 'Orta', 'İyi', 'Mükemmel']
credit_risk['fico_category'] = pd.cut(credit_risk['fico_range_low'], bins=bins, labels=labels)

# Kredi Tutarlarını Gruplandırma yapıldı
loan_bins = [0, 5000, 15000, 25000, 40000]
loan_labels = ['Küçük Ölçekli', 'Orta Ölçekli', 'Büyük Ölçekli', 'Çok Büyük']
loan_details['loan_size_category'] = pd.cut(loan_details['loan_amnt'], bins=loan_bins, labels=loan_labels)

print("✅ Looker filtreleri için sayısal gruplandırmalar (binning) yapıldı.")

✅ Looker filtreleri için sayısal gruplandırmalar (binning) yapıldı.


**6. "Days to Default" veya "Credit Age" Hesaplamaları**
Looker'da iki tarih arasındaki farkı alıp bunu sayıya çevirirken hata almamak için

In [ ]:
# Kredinin verildiği tarih ile ilk kredi hesabı arasındaki fark (Kredi Geçmişi Yaşı)
credit_risk['credit_history_age_years'] = (loan_details['issue_d'] - credit_risk['earliest_cr_line']).dt.days / 365

# Negatif değerler varsa (veri hatası) onları 0'a eşitlendi
credit_risk['credit_history_age_years'] = credit_risk['credit_history_age_years'].apply(lambda x: x if x > 0 else 0)

print("✅ Kredi geçmişi yaşı (Yıl bazında) hesaplandı.")

✅ Kredi geçmişi yaşı (Yıl bazında) hesaplandı.


**7. Sektörel / İş Unvanı Gruplandırması (emp_title)**
emp_title sütununda binlerce farklı unvan var. Looker'da bir tabloda "Manager"ı ararken "Project Manager", "Regional Manager" gibi onlarca farklı satır çıkacağı için bunu Python'da ana kategorilere çektim.

In [ ]:
# İş unvanlarını basitleştirelim
def categorize_title(title):
    title = str(title).lower()
    if 'manager' in title or 'director' in title or 'vp' in title: return 'Yönetici'
    elif 'teacher' in title or 'professor' in title: return 'Eğitim'
    elif 'nurse' in title or 'doctor' in title or 'medical' in title: return 'Sağlık'
    elif 'engineer' in title or 'analyst' in title or 'developer' in title: return 'Teknik/Analiz'
    elif 'sales' in title or 'marketing' in title: return 'Satış/Pazarlama'
    elif 'driver' in title or 'truck' in title: return 'Lojistik/Ulaşım'
    else: return 'Diğer'

borrowers['job_category'] = borrowers['emp_title'].apply(categorize_title)

print("✅ İş unvanları 7 ana kategoriye indirgendi.")

In [ ]:
import pandas as pd
import numpy as np
import pandas_gbq
from google.colab import drive
from google.colab import auth# 1. Bağlantılar ve Yetkilendirme
drive.mount('/content/drive')
auth.authenticate_user()# Proje Bilgileri
project_id = 'promising-haiku-476413-v8'
dataset_id = 'lending_club_project'
file_path = '/content/drive/MyDrive/accepted_2007_to_2018Q4.csv'# 2. Gelişmiş Temizlik ve Metrik Fonksiyonu
def process_chunk(chunk):
    # Sayısal Format Düzeltmeleri
    chunk['int_rate'] = chunk['int_rate'].astype(str).str.replace('%', '').replace('nan', np.nan).astype(float)
    chunk['term'] = chunk['term'].astype(str).str.extract(r'(\d+)').astype(float)    # Tarih Dönüşümleri
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    chunk['earliest_cr_line'] = pd.to_datetime(chunk['earliest_cr_line'], format='%b-%Y', errors='coerce')    # YENİ METRİKLER: Kredi/Gelir Oranı ve Net Kayıp
    chunk['loan_to_income'] = chunk['loan_amnt'] / chunk['annual_inc']
    # Net Kayıp: Verilen para - Toplam geri alınan (Sadece Charged Off durumu için anlamlıdır)
    chunk['net_loss'] = chunk['loan_amnt'] - chunk['total_pymnt']    # İş Unvanı Kategorizasyonu (Text Mining)
    def categorize_title(title):
        title = str(title).lower()
        if any(x in title for x in ['manager', 'director', 'vp', 'supervisor', 'chief']): return 'Yönetici'
        if any(x in title for x in ['engineer', 'analyst', 'developer', 'it']): return 'Teknik/Analiz'
        if any(x in title for x in ['nurse', 'doctor', 'medical', 'health']): return 'Sağlık'
        if any(x in title for x in ['teacher', 'professor', 'instructor', 'academic']): return 'Eğitim'
        if any(x in title for x in ['driver', 'truck', 'delivery', 'warehouse']): return 'Lojistik/Ulaşım'
        if any(x in title for x in ['sales', 'marketing', 'account']): return 'Satış/Pazarlama'
        return 'Diğer'    chunk['job_category'] = chunk['emp_title'].apply(categorize_title)    # Eyalet ve Ev Sahipliği Standardizasyonu
    chunk['addr_state'] = chunk['addr_state'].str.upper()
    chunk['home_ownership'] = chunk['home_ownership'].replace(['ANY', 'NONE'], 'OTHER')    return chunk# 3. Ana İşleme Döngüsü (Parçalı Yükleme)
chunk_size = 250000 # Her seferde 250 bin satır işler
is_first_chunk = Truetry:
    print(":hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...")    # Parça parça oku
    for chunk in pd.read_csv(file_path, low_memory=False, chunksize=chunk_size):        # Temizlik ve Metrikleri Uygula
        processed_chunk = process_chunk(chunk)        # Tabloları Oluştur
        tables = {
            'borrowers': processed_chunk[['id', 'job_category', 'emp_length', 'home_ownership', 'annual_inc', 'addr_state']],
            'loan_details': processed_chunk[['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'issue_d', 'loan_to_income']],
            'credit_risk': processed_chunk[['id', 'fico_range_low', 'earliest_cr_line', 'delinq_2yrs']],
            'financial_health': processed_chunk[['id', 'dti', 'revol_util']],
            'loan_performance': processed_chunk[['id', 'loan_status', 'total_pymnt', 'net_loss']]
        }        # Her tabloyu BigQuery'ye gönder
        for table_name, table_df in tables.items():
            # İlk parçada tabloyu sıfırla (replace), sonrakilerde üzerine ekle (append)
            mode = 'replace' if is_first_chunk else 'append'            pandas_gbq.to_gbq(
                table_df,
                f"{dataset_id}.{table_name}",
                project_id=project_id,
                if_exists=mode,
                progress_bar=False
            )        print(f":white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: {mode})")
        is_first_chunk = False    print("\n:sparkles: EFSANE! 2.2 Milyon satırın tamamı, tüm temizlik ve metriklerle 5 ayrı tabloya aktarıldı.")except Exception as e:
    print(f":x: Bir hata oluştu: {e}")


SyntaxError: invalid syntax (ipython-input-3428458041.py, line 28)

# Task
Kullanıcının verdiği büyük LendingClub veri setini, temizleyip, dönüştürerek ve yeni metrikler oluşturarak 5 mantıksal tabloya bölme ve bu tabloları `promising-haiku-476413-v8.lending_club_project` BigQuery veri tabanına yükleme işlemini tamamla.

## Fix SyntaxError in Oa7bE2H_p3bw

### Subtask:
Adjust the indentation and line breaks in cell `Oa7bE2H_p3bw` to resolve the `SyntaxError: invalid syntax`. The line assigning `job_category` and subsequent data cleaning steps should be placed on new lines after the `categorize_title` function definition, at the correct indentation level within the `process_chunk` function.


**Reasoning**:
To fix the `SyntaxError`, I need to correct the line breaks and indentation in the `process_chunk` function within cell `Oa7bE2H_p3bw`. The `return` statement and the assignment to `chunk['job_category']` were on the same line, causing the error. I will rewrite the entire code for that cell to ensure all changes are applied correctly.



In [ ]:
import pandas as pd
import numpy as np
import pandas_gbq
from google.colab import drive
from google.colab import auth

# 1. Bağlantılar ve Yetkilendirme
drive.mount('/content/drive')
auth.authenticate_user()

# Proje Bilgileri
project_id = 'promising-haiku-476413-v8'
dataset_id = 'lending_club_project'
file_path = '/content/drive/MyDrive/accepted_2007_to_2018Q4.csv'

# 2. Gelişmiş Temizlik ve Metrik Fonksiyonu
def process_chunk(chunk):
    # Sayısal Format Düzeltmeleri
    chunk['int_rate'] = chunk['int_rate'].astype(str).str.replace('%', '').replace('nan', np.nan).astype(float)
    chunk['term'] = chunk['term'].astype(str).str.extract(r'(\d+)').astype(float)

    # Tarih Dönüşümleri
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    chunk['earliest_cr_line'] = pd.to_datetime(chunk['earliest_cr_line'], format='%b-%Y', errors='coerce')

    # YENİ METRİKLER: Kredi/Gelir Oranı ve Net Kayıp
    chunk['loan_to_income'] = chunk['loan_amnt'] / chunk['annual_inc']
    # Net Kayıp: Verilen para - Toplam geri alınan (Sadece Charged Off durumu için anlamlıdır)
    chunk['net_loss'] = chunk['loan_amnt'] - chunk['total_pymnt']

    # İş Unvanı Kategorizasyonu (Text Mining)
    def categorize_title(title):
        title = str(title).lower()
        if any(x in title for x in ['manager', 'director', 'vp', 'supervisor', 'chief']): return 'Yönetici'
        if any(x in title for x in ['engineer', 'analyst', 'developer', 'it']): return 'Teknik/Analiz'
        if any(x in title for x in ['nurse', 'doctor', 'medical', 'health']): return 'Sağlık'
        if any(x in title for x in ['teacher', 'professor', 'instructor', 'academic']): return 'Eğitim'
        if any(x in title for x in ['driver', 'truck', 'delivery', 'warehouse']): return 'Lojistik/Ulaşım'
        if any(x in title for x in ['sales', 'marketing', 'account']): return 'Satış/Pazarlama'
        return 'Diğer'

    chunk['job_category'] = chunk['emp_title'].apply(categorize_title)

    # Eyalet ve Ev Sahipliği Standardizasyonu
    chunk['addr_state'] = chunk['addr_state'].str.upper()
    chunk['home_ownership'] = chunk['home_ownership'].replace(['ANY', 'NONE'], 'OTHER')

    return chunk

# 3. Ana İşleme Döngüsü (Parçalı Yükleme)
chunk_size = 250000 # Her seferde 250 bin satır işler
is_first_chunk = True
try:
    print(":hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...")

    # Parça parça oku
    for chunk in pd.read_csv(file_path, low_memory=False, chunksize=chunk_size):
        # Temizlik ve Metrikleri Uygula
        processed_chunk = process_chunk(chunk)

        # Tabloları Oluştur
        tables = {
            'borrowers': processed_chunk[['id', 'job_category', 'emp_length', 'home_ownership', 'annual_inc', 'addr_state']],
            'loan_details': processed_chunk[['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'issue_d', 'loan_to_income']],
            'credit_risk': processed_chunk[['id', 'fico_range_low', 'earliest_cr_line', 'delinq_2yrs']],
            'financial_health': processed_chunk[['id', 'dti', 'revol_util']],
            'loan_performance': processed_chunk[['id', 'loan_status', 'total_pymnt', 'net_loss']]
        }

        # Her tabloyu BigQuery'ye gönder
        for table_name, table_df in tables.items():
            # İlk parçada tabloyu sıfırla (replace), sonrakilerde üzerine ekle (append)
            mode = 'replace' if is_first_chunk else 'append'

            pandas_gbq.to_gbq(
                table_df,
                f"{dataset_id}.{table_name}",
                project_id=project_id,
                if_exists=mode,
                progress_bar=False
            )

        print(f":white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: {mode})")
        is_first_chunk = False

    print("\n:sparkles: EFSANE! 2.2 Milyon satırın tamamı, tüm temizlik ve metriklerle 5 ayrı tabloya aktarıldı.")

except Exception as e:
    print(f":x: Bir hata oluştu: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
:hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...
:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: replace)
:x: Bir hata oluştu: Could not convert DataFrame to Parquet.


## Integrate emp_length cleaning into process_chunk

### Subtask:
Modify the `process_chunk` function to properly clean and type-cast the `emp_length` column, which is likely causing the 'Could not convert DataFrame to Parquet' error during BigQuery upload.


**Reasoning**:
The user wants to integrate the `emp_length` cleaning logic into the `process_chunk` function. I will update the existing `process_chunk` function in cell `342e030a` to include the `clean_emp_length` helper function and apply it to the `emp_length` column, ensuring it's converted to a numeric type.



In [ ]:
import pandas as pd
import numpy as np
import pandas_gbq
from google.colab import drive
from google.colab import auth

# 1. Bağlantılar ve Yetkilendirme
drive.mount('/content/drive')
auth.authenticate_user()

# Proje Bilgileri
project_id = 'promising-haiku-476413-v8'
dataset_id = 'lending_club_project'
file_path = '/content/drive/MyDrive/accepted_2007_to_2018Q4.csv'

# 2. Gelişmiş Temizlik ve Metrik Fonksiyonu
def process_chunk(chunk):
    # Sayısal Format Düzeltmeleri
    chunk['int_rate'] = chunk['int_rate'].astype(str).str.replace('%', '').replace('nan', np.nan).astype(float)
    chunk['term'] = chunk['term'].astype(str).str.extract(r'(\d+)').astype(float)

    # emp_length temizliği
    def clean_emp_length(val):
        val = str(val).lower() # Değeri metne çevir ve küçük harf yap
        if val == 'nan' or val == 'none' or val == '':
            return 0
        if '10+' in val:
            return 10
        if '< 1' in val:
            return 0
        # İçindeki rakamları ayıkla
        digits = ''.join(filter(str.isdigit, val))
        return int(digits) if digits else 0

    chunk['emp_length'] = chunk['emp_length'].apply(clean_emp_length).astype(int)

    # Tarih Dönüşümleri
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    chunk['earliest_cr_line'] = pd.to_datetime(chunk['earliest_cr_line'], format='%b-%Y', errors='coerce')

    # YENİ METRİKLER: Kredi/Gelir Oranı ve Net Kayıp
    chunk['loan_to_income'] = chunk['loan_amnt'] / chunk['annual_inc']
    # Net Kayıp: Verilen para - Toplam geri alınan (Sadece Charged Off durumu için anlamlıdır)
    chunk['net_loss'] = chunk['loan_amnt'] - chunk['total_pymnt']

    # İş Unvanı Kategorizasyonu (Text Mining)
    def categorize_title(title):
        title = str(title).lower()
        if any(x in title for x in ['manager', 'director', 'vp', 'supervisor', 'chief']): return 'Yönetici'
        if any(x in title for x in ['engineer', 'analyst', 'developer', 'it']): return 'Teknik/Analiz'
        if any(x in title for x in ['nurse', 'doctor', 'medical', 'health']): return 'Sağlık'
        if any(x in title for x in ['teacher', 'professor', 'instructor', 'academic']): return 'Eğitim'
        if any(x in title for x in ['driver', 'truck', 'delivery', 'warehouse']): return 'Lojistik/Ulaşım'
        if any(x in title for x in ['sales', 'marketing', 'account']): return 'Satış/Pazarlama'
        return 'Diğer'

    chunk['job_category'] = chunk['emp_title'].apply(categorize_title)

    # Eyalet ve Ev Sahipliği Standardizasyonu
    chunk['addr_state'] = chunk['addr_state'].str.upper()
    chunk['home_ownership'] = chunk['home_ownership'].replace(['ANY', 'NONE'], 'OTHER')

    return chunk

# 3. Ana İşleme Döngüsü (Parçalı Yükleme)
chunk_size = 250000 # Her seferde 250 bin satır işler
is_first_chunk = True
try:
    print(":hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...")

    # Parça parça oku
    for chunk in pd.read_csv(file_path, low_memory=False, chunksize=chunk_size):
        # Temizlik ve Metrikleri Uygula
        processed_chunk = process_chunk(chunk)

        # Tabloları Oluştur
        tables = {
            'borrowers': processed_chunk[['id', 'job_category', 'emp_length', 'home_ownership', 'annual_inc', 'addr_state']],
            'loan_details': processed_chunk[['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'issue_d', 'loan_to_income']],
            'credit_risk': processed_chunk[['id', 'fico_range_low', 'earliest_cr_line', 'delinq_2yrs']],
            'financial_health': processed_chunk[['id', 'dti', 'revol_util']],
            'loan_performance': processed_chunk[['id', 'loan_status', 'total_pymnt', 'net_loss']]
        }

        # Her tabloyu BigQuery'ye gönder
        for table_name, table_df in tables.items():
            # İlk parçada tabloyu sıfırla (replace), sonrakilerde üzerine ekle (append)
            mode = 'replace' if is_first_chunk else 'append'

            pandas_gbq.to_gbq(
                table_df,
                f"{dataset_id}.{table_name}",
                project_id=project_id,
                if_exists=mode,
                progress_bar=False
            )

        print(f":white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: {mode})")
        is_first_chunk = False

    print("\n:sparkles: EFSANE! 2.2 Milyon satırın tamamı, tüm temizlik ve metriklerle 5 ayrı tabloya aktarıldı.")

except Exception as e:
    print(f":x: Bir hata oluştu: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
:hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...
:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: replace)
:x: Bir hata oluştu: Could not convert DataFrame to Parquet.


**Reasoning**:
The 'Could not convert DataFrame to Parquet' error often indicates issues with data types, specifically NaN values in integer columns or `inf` values in float columns. I will enhance the `process_chunk` function to robustly handle NaN and Inf values in all numeric columns before they are sent to BigQuery, converting them to appropriate types (e.g., filling NaNs with 0).



In [ ]:
import pandas as pd
import numpy as np
import pandas_gbq
from google.colab import drive
from google.colab import auth

# 1. Bağlantılar ve Yetkilendirme
drive.mount('/content/drive')
auth.authenticate_user()

# Proje Bilgileri
project_id = 'promising-haiku-476413-v8'
dataset_id = 'lending_club_project'
file_path = '/content/drive/MyDrive/accepted_2007_to_2018Q4.csv'

# 2. Gelişmiş Temizlik ve Metrik Fonksiyonu
def process_chunk(chunk):
    # Sayısal Format Düzeltmeleri
    chunk['int_rate'] = chunk['int_rate'].astype(str).str.replace('%', '').replace('nan', np.nan).astype(float)
    chunk['term'] = chunk['term'].astype(str).str.extract(r'(\d+)').astype(float)

    # emp_length temizliği
    def clean_emp_length(val):
        val = str(val).lower() # Değeri metne çevir ve küçük harf yap
        if val == 'nan' or val == 'none' or val == '':
            return 0
        if '10+' in val:
            return 10
        if '< 1' in val:
            return 0
        # İçindeki rakamları ayıkla
        digits = ''.join(filter(str.isdigit, val))
        return int(digits) if digits else 0

    chunk['emp_length'] = chunk['emp_length'].apply(clean_emp_length).astype(int)

    # Tarih Dönüşümleri
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    chunk['earliest_cr_line'] = pd.to_datetime(chunk['earliest_cr_line'], format='%b-%Y', errors='coerce')

    # YENİ METRİKLER: Kredi/Gelir Oranı ve Net Kayıp
    # Handle division by zero for annual_inc which would result in inf
    chunk['loan_to_income'] = chunk['loan_amnt'] / chunk['annual_inc']
    chunk['loan_to_income'] = chunk['loan_to_income'].replace([np.inf, -np.inf], np.nan)

    # Net Kayıp: Verilen para - Toplam geri alınan (Sadece Charged Off durumu için anlamlıdır)
    chunk['net_loss'] = chunk['loan_amnt'] - chunk['total_pymnt']

    # İş Unvanı Kategorizasyonu (Text Mining)
    def categorize_title(title):
        title = str(title).lower()
        if any(x in title for x in ['manager', 'director', 'vp', 'supervisor', 'chief']): return 'Yönetici'
        if any(x in title for x in ['engineer', 'analyst', 'developer', 'it']): return 'Teknik/Analiz'
        if any(x in title for x in ['nurse', 'doctor', 'medical', 'health']): return 'Sağlık'
        if any(x in title for x in ['teacher', 'professor', 'instructor', 'academic']): return 'Eğitim'
        if any(x in title for x in ['driver', 'truck', 'delivery', 'warehouse']): return 'Lojistik/Ulaşım'
        if any(x in title for x in ['sales', 'marketing', 'account']): return 'Satış/Pazarlama'
        return 'Diğer'

    chunk['job_category'] = chunk['emp_title'].apply(categorize_title)

    # Eyalet ve Ev Sahipliği Standardizasyonu
    chunk['addr_state'] = chunk['addr_state'].str.upper()
    chunk['home_ownership'] = chunk['home_ownership'].replace(['ANY', 'NONE'], 'OTHER')

    # Son Kontroller: NaN değerlerini doldurma (BigQuery uyumluluğu için)
    numeric_cols_to_fill_zero = [
        'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment',
        'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
        'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'total_acc', 'total_pymnt',
        'recoveries', 'emp_length', 'loan_to_income', 'net_loss', 'delinq_2yrs'
    ]
    for col in numeric_cols_to_fill_zero:
        if col in chunk.columns:
            if pd.api.types.is_numeric_dtype(chunk[col]):
                chunk[col] = chunk[col].fillna(0)

    # Fill NaT for date columns, for example, with a default date or handle them as nullable in BigQuery
    # For this exercise, we will just ensure they are datetime objects, pandas_gbq usually handles NaT for nullable date types.
    # If BigQuery schema expects non-nullable, a default date like pd.Timestamp('1970-01-01') would be needed.

    return chunk

# 3. Ana İşleme Döngüsü (Parçalı Yükleme)
chunk_size = 250000 # Her seferde 250 bin satır işler
is_first_chunk = True
try:
    print(":hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...")

    # Parça parça oku
    for chunk in pd.read_csv(file_path, low_memory=False, chunksize=chunk_size):
        # Temizlik ve Metrikleri Uygula
        processed_chunk = process_chunk(chunk)

        # Tabloları Oluştur
        tables = {
            'borrowers': processed_chunk[['id', 'job_category', 'emp_length', 'home_ownership', 'annual_inc', 'addr_state']],
            'loan_details': processed_chunk[['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'issue_d', 'loan_to_income']],
            'credit_risk': processed_chunk[['id', 'fico_range_low', 'earliest_cr_line', 'delinq_2yrs']],
            'financial_health': processed_chunk[['id', 'dti', 'revol_util']],
            'loan_performance': processed_chunk[['id', 'loan_status', 'total_pymnt', 'net_loss']]
        }

        # Her tabloyu BigQuery'ye gönder
        for table_name, table_df in tables.items():
            # İlk parçada tabloyu sıfırla (replace), sonrakilerde üzerine ekle (append)
            mode = 'replace' if is_first_chunk else 'append'

            # Ensure 'id' column is of a type BigQuery can handle, e.g., string or int
            table_df['id'] = table_df['id'].astype(str)

            pandas_gbq.to_gbq(
                table_df,
                f"{dataset_id}.{table_name}",
                project_id=project_id,
                if_exists=mode,
                progress_bar=False
            )

        print(f":white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: {mode})")
        is_first_chunk = False

    print("\n:sparkles: EFSANE! 2.2 Milyon satırın tamamı, tüm temizlik ve metriklerle 5 ayrı tabloya aktarıldı.")

except Exception as e:
    print(f":x: Bir hata oluştu: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
:hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...


/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: replace)


/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: append)


...

:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: append)


/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: append)


/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df['id'] = table_df['id'].astype(str)
/tmp/ipython-input-3180238407.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

:white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: append)

:sparkles: EFSANE! 2.2 Milyon satırın tamamı, tüm temizlik ve metriklerle 5 ayrı tabloya aktarıldı.


**Reasoning**:
The `SettingWithCopyWarning` indicates that `table_df` is a view of a slice of another DataFrame, and modifying it directly can lead to unexpected behavior. To resolve this, I will explicitly create a copy of the DataFrame when assigning to `table_df` within the loop to ensure modifications are made on an independent copy.



In [ ]:
import pandas as pd
import numpy as np
import pandas_gbq
from google.colab import drive
from google.colab import auth

# 1. Bağlantılar ve Yetkilendirme
drive.mount('/content/drive')
auth.authenticate_user()

# Proje Bilgileri
project_id = 'promising-haiku-476413-v8'
dataset_id = 'lending_club_project'
file_path = '/content/drive/MyDrive/accepted_2007_to_2018Q4.csv'

# 2. Gelişmiş Temizlik ve Metrik Fonksiyonu
def process_chunk(chunk):
    # Sayısal Format Düzeltmeleri
    chunk['int_rate'] = chunk['int_rate'].astype(str).str.replace('%', '').replace('nan', np.nan).astype(float)
    chunk['term'] = chunk['term'].astype(str).str.extract(r'(\d+)').astype(float)

    # emp_length temizliği
    def clean_emp_length(val):
        val = str(val).lower() # Değeri metne çevir ve küçük harf yap
        if val == 'nan' or val == 'none' or val == '':
            return 0
        if '10+' in val:
            return 10
        if '< 1' in val:
            return 0
        # İçindeki rakamları ayıkla
        digits = ''.join(filter(str.isdigit, val))
        return int(digits) if digits else 0

    chunk['emp_length'] = chunk['emp_length'].apply(clean_emp_length).astype(int)

    # Tarih Dönüşümleri
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    chunk['earliest_cr_line'] = pd.to_datetime(chunk['earliest_cr_line'], format='%b-%Y', errors='coerce')

    # YENİ METRİKLER: Kredi/Gelir Oranı ve Net Kayıp
    # Handle division by zero for annual_inc which would result in inf
    chunk['loan_to_income'] = chunk['loan_amnt'] / chunk['annual_inc']
    chunk['loan_to_income'] = chunk['loan_to_income'].replace([np.inf, -np.inf], np.nan)

    # Net Kayıp: Verilen para - Toplam geri alınan (Sadece Charged Off durumu için anlamlıdır)
    chunk['net_loss'] = chunk['loan_amnt'] - chunk['total_pymnt']

    # İş Unvanı Kategorizasyonu (Text Mining)
    def categorize_title(title):
        title = str(title).lower()
        if any(x in title for x in ['manager', 'director', 'vp', 'supervisor', 'chief']): return 'Yönetici'
        if any(x in title for x in ['engineer', 'analyst', 'developer', 'it']): return 'Teknik/Analiz'
        if any(x in title for x in ['nurse', 'doctor', 'medical', 'health']): return 'Sağlık'
        if any(x in title for x in ['teacher', 'professor', 'instructor', 'academic']): return 'Eğitim'
        if any(x in title for x in ['driver', 'truck', 'delivery', 'warehouse']): return 'Lojistik/Ulaşım'
        if any(x in title for x in ['sales', 'marketing', 'account']): return 'Satış/Pazarlama'
        return 'Diğer'

    chunk['job_category'] = chunk['emp_title'].apply(categorize_title)

    # Eyalet ve Ev Sahipliği Standardizasyonu
    chunk['addr_state'] = chunk['addr_state'].str.upper()
    chunk['home_ownership'] = chunk['home_ownership'].replace(['ANY', 'NONE'], 'OTHER')

    # Son Kontroller: NaN değerlerini doldurma (BigQuery uyumluluğu için)
    numeric_cols_to_fill_zero = [
        'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment',
        'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
        'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'total_acc', 'total_pymnt',
        'recoveries', 'emp_length', 'loan_to_income', 'net_loss', 'delinq_2yrs'
    ]
    for col in numeric_cols_to_fill_zero:
        if col in chunk.columns:
            if pd.api.types.is_numeric_dtype(chunk[col]):
                chunk[col] = chunk[col].fillna(0)

    # Fill NaT for date columns, for example, with a default date or handle them as nullable in BigQuery
    # For this exercise, we will just ensure they are datetime objects, pandas_gbq usually handles NaT for nullable date types.
    # If BigQuery schema expects non-nullable, a default date like pd.Timestamp('1970-01-01') would be needed.

    return chunk

# 3. Ana İşleme Döngüsü (Parçalı Yükleme)
chunk_size = 250000 # Her seferde 250 bin satır işler
is_first_chunk = True
try:
    print(":hourglass_flowing_sand: Büyük veri işleme ve BigQuery aktarımı başladı (Yaklaşık 5-10 dk sürebilir)...")

    # Parça parça oku
    for chunk in pd.read_csv(file_path, low_memory=False, chunksize=chunk_size):
        # Temizlik ve Metrikleri Uygula
        processed_chunk = process_chunk(chunk)

        # Tabloları Oluştur. .copy() ekleyerek SettingWithCopyWarning'i engelle
        tables = {
            'borrowers': processed_chunk[['id', 'job_category', 'emp_length', 'home_ownership', 'annual_inc', 'addr_state']].copy(),
            'loan_details': processed_chunk[['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'issue_d', 'loan_to_income']].copy(),
            'credit_risk': processed_chunk[['id', 'fico_range_low', 'earliest_cr_line', 'delinq_2yrs']].copy(),
            'financial_health': processed_chunk[['id', 'dti', 'revol_util']].copy(),
            'loan_performance': processed_chunk[['id', 'loan_status', 'total_pymnt', 'net_loss']].copy()
        }

        # Her tabloyu BigQuery'ye gönder
        for table_name, table_df in tables.items():
            # İlk parçada tabloyu sıfırla (replace), sonrakilerde üzerine ekle (append)
            mode = 'replace' if is_first_chunk else 'append'

            # Ensure 'id' column is of a type BigQuery can handle, e.g., string or int
            table_df['id'] = table_df['id'].astype(str)

            pandas_gbq.to_gbq(
                table_df,
                f"{dataset_id}.{table_name}",
                project_id=project_id,
                if_exists=mode,
                progress_bar=False
            )

        print(f":white_check_mark: Bir bölüm daha aktarıldı (Mevcut mod: {mode})")
        is_first_chunk = False

    print("\n:sparkles: EFSANE! 2.2 Milyon satırın tamamı, tüm temizlik ve metriklerle 5 ayrı tabloya aktarıldı.")

except Exception as e:
    print(f":x: Bir hata oluştu: {e}")

**1. Hedef Değişkeni (Target) Düzenleme**